In [ ]:
!pip install -q --no-index --find-links /kaggle/input/datasets/dennisfong/nvidia-nemotron-offline-packages/offline_packages datasets trl --ignore-installed

In [ ]:
import datasets
import trl
print("Successfully imported datasets version:", datasets.__version__)
print("Successfully imported trl version:", trl.__version__)

In [ ]:
# Stub out optional modules that trl tries to import but doesn't need for GRPO
import sys, types, importlib
_stub_names = [
    "mergekit", "mergekit.config", "mergekit.merge", "mergekit.common",
    "llm_blender", "liger_kernel", "liger_kernel.transformers",
    "weave",
]
for mod_name in _stub_names:
    if mod_name not in sys.modules:
        mod = types.ModuleType(mod_name)
        mod.__version__ = "0.0.0"
        mod.__spec__ = importlib.machinery.ModuleSpec(mod_name, None)
        sys.modules[mod_name] = mod

import trl.import_utils as _trl_iu
_trl_iu.is_mergekit_available = lambda: False
_trl_iu.is_weave_available = lambda: False
print("Optional modules stubbed, ready for GRPOTrainer import")

In [ ]:
import sys
import types

for _mod_name in [
    'mamba_ssm.modules.mamba3',
    'mamba_ssm.ops.cute',
    'mamba_ssm.ops.cute.mamba3',
    'mamba_ssm.ops.cute.mamba3.mamba3_step_fn',
]:
    sys.modules[_mod_name] = types.ModuleType(_mod_name)
# Give mamba3 a dummy Mamba3 class so the __init__ import doesn't fail
sys.modules['mamba_ssm.modules.mamba3'].Mamba3 = None

In [ ]:
import os
import sys
import stat
import shutil
import gc
import re
import math
import json
import zipfile
from decimal import Decimal, ROUND_HALF_UP
from itertools import combinations

import polars as pl
import torch
import torch.nn.functional as F
import kagglehub
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, PeftModel, TaskType
from trl import SFTTrainer, SFTConfig, GRPOTrainer, GRPOConfig

In [ ]:
# === Triton fixes ===
def _pure_rmsnorm_fn(x, weight, bias=None, z=None, eps=1e-5,
                     group_size=None, norm_before_gate=True, upcast=True):
    dtype = x.dtype
    if upcast:
        x = x.float()
    variance = x.pow(2).mean(-1, keepdim=True)
    x_normed = x * torch.rsqrt(variance + eps)
    out = x_normed * weight.float()
    if bias is not None:
        out = out + bias.float()
    if z is not None:
        out = out * F.silu(z.float())
    return out.to(dtype)

for name, mod in list(sys.modules.items()):
    if hasattr(mod, 'rmsnorm_fn'):
        mod.rmsnorm_fn = _pure_rmsnorm_fn

src = "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/triton/backends/nvidia/bin/ptxas-blackwell"
dst = "/tmp/ptxas-blackwell"
if os.path.exists(src):
    shutil.copy2(src, dst)
    os.chmod(dst, os.stat(dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)

    import triton.backends.nvidia as nv_backend
    src_bin = os.path.join(os.path.dirname(nv_backend.__file__), "bin")
    dst_bin = "/tmp/triton_nvidia_bin"
    shutil.copytree(src_bin, dst_bin, dirs_exist_ok=True)
    for f in os.listdir(dst_bin):
        fp = os.path.join(dst_bin, f)
        if os.path.isfile(fp):
            os.chmod(fp, os.stat(fp).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)

    nv_backend.__file__ = os.path.join(dst_bin, "..", "__init__.py")
    os.environ["TRITON_PTXAS_PATH"] = dst
    print("Triton ptxas fix applied.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ═══════════════════════════════════════════════════════════════════════════════

# ── Resume control ──
SKIP_SFT = False          # True = skip SFT, load adapter from checkpoint, go straight to GRPO
SFT_CHECKPOINT_PATH = "/kaggle/input/datasets/zuhairsan/nemotron-lora-adapter1/nemotron-lora-adapter"

# ── SFT Configuration ──
SUBSAMPLE_SIZE = 9500     # use full dataset for SFT
LORA_RANK = 32
MAX_SEQ_LEN = 2048
NUM_EPOCHS = 1
GRAD_ACCUM = 4
LR = 2e-4
OUTPUT_DIR = "/kaggle/working/adapter"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── GRPO Configuration ──
GRPO_SUBSAMPLE_SIZE = 500
GRPO_NUM_GENERATIONS = 2   # was 4
GRPO_MAX_COMPLETION = 1024 # was 2048
GRPO_LR = 5e-6
GRPO_EPOCHS = 1
GRPO_TEMPERATURE = 0.7

# ── Data ──
MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
train_df = pl.read_csv('/kaggle/input/nvidia-nemotron-3-reasoning-challenge/train.csv')
print(f"Total training samples: {len(train_df)}")

if SUBSAMPLE_SIZE < len(train_df):
    train_df = train_df.sample(n=SUBSAMPLE_SIZE, seed=42)
    print(f"Subsampled to {SUBSAMPLE_SIZE}")

train_pd = train_df.to_pandas()
hf_dataset = Dataset.from_pandas(train_pd)

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Config ready. SKIP_SFT={SKIP_SFT}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# DETERMINISTIC SOLVERS — Generate exact answers + reasoning traces
# ═══════════════════════════════════════════════════════════════════════════════

BOXED_INSTRUCTION = "\nPut your final answer inside \\boxed{}."

def _round2_candidates(val):
    candidates = set()
    candidates.add(f"{round(val, 2):.2f}")
    d = Decimal(str(val)).quantize(Decimal('0.01'), rounding=ROUND_HALF_UP)
    candidates.add(str(d))
    candidates.add(f"{math.floor(val * 100) / 100:.2f}")
    candidates.add(f"{math.ceil(val * 100) / 100:.2f}")
    for c in list(candidates):
        if c.endswith('0') and '.' in c:
            candidates.add(c.rstrip('0').rstrip('.'))
    return candidates

def solve_gravity(prompt, answer):
    pairs = re.findall(r't\s*=\s*([\d.]+)\s*s.*?distance\s*=\s*([\d.]+)\s*m', prompt)
    query_t_m = re.search(r'falling distance for t\s*=\s*([\d.]+)\s*s', prompt)
    if not pairs or not query_t_m:
        return None, None
    gs = [2 * float(d) / (float(t)**2) for t, d in pairs]
    g_avg = sum(gs) / len(gs)
    t_q = float(query_t_m.group(1))
    best_g, val = g_avg, 0.5 * g_avg * t_q**2
    if answer not in _round2_candidates(val):
        for gi in gs:
            vi = 0.5 * gi * t_q**2
            if answer in _round2_candidates(vi):
                best_g, val = gi, vi
                break
    predicted = answer if answer in _round2_candidates(val) else f"{val:.2f}"
    g_lines = "\n".join(f"  Example {i+1}: g = 2*{d}/{t}^2 = {2*float(d)/float(t)**2:.4f}" for i, (t, d) in enumerate(pairs))
    cot = f"Using d = 0.5*g*t^2, so g = 2d/t^2.\n\nComputing g:\n{g_lines}\n\ng = {best_g:.4f}\n\nFor t={t_q}s: d = 0.5*{best_g:.4f}*{t_q**2:.2f} = {predicted}\n\n\\boxed{{{predicted}}}"
    return predicted, cot

def solve_unit_conversion(prompt, answer):
    pairs = re.findall(r'([\d.]+)\s*m\s+becomes\s+([\d.]+)', prompt)
    query_m = re.search(r'convert the following measurement:\s*([\d.]+)\s*m', prompt)
    if not pairs or not query_m:
        return None, None
    ratios = [float(out) / float(inp) for inp, out in pairs]
    ratio = sum(ratios) / len(ratios)
    q = float(query_m.group(1))
    val = ratio * q
    if answer not in _round2_candidates(val):
        for r in ratios:
            if answer in _round2_candidates(r * q):
                ratio, val = r, r * q
                break
    predicted = answer if answer in _round2_candidates(val) else f"{val:.2f}"
    cot = f"Finding conversion factor from examples.\n\nRatios: {[f'{float(o)/float(i):.4f}' for i,o in pairs]}\nFactor = {ratio:.4f}\n\nResult = {ratio:.4f} * {q} = {predicted}\n\n\\boxed{{{predicted}}}"
    return predicted, cot

def _int_to_roman(num):
    vals = [1000,900,500,400,100,90,50,40,10,9,5,4,1]
    syms = ['M','CM','D','CD','C','XC','L','XL','X','IX','V','IV','I']
    r = ''
    for v, s in zip(vals, syms):
        while num >= v: r += s; num -= v
    return r

def _roman_to_int(s):
    vals = {'I':1,'V':5,'X':10,'L':50,'C':100,'D':500,'M':1000}
    total = 0
    for i in range(len(s)):
        if i+1 < len(s) and vals.get(s[i],0) < vals.get(s[i+1],0):
            total -= vals.get(s[i],0)
        else:
            total += vals.get(s[i],0)
    return total

def solve_base_conversion(prompt, answer):
    examples = re.findall(r'(\d+)\s*->\s*([A-Z]+)', prompt)
    if examples and all(_int_to_roman(int(n)) == r for n, r in examples):
        query = re.search(r'(?:write|convert)\s+(?:the\s+)?number\s+(\d+)', prompt)
        if query:
            num = int(query.group(1))
            predicted = _int_to_roman(num)
            cot = f"Decimal to Roman numeral conversion.\n\nConverting {num}: {predicted}\n\n\\boxed{{{predicted}}}"
            return predicted, cot
    examples_rev = re.findall(r'([A-Z]+)\s*->\s*(\d+)', prompt)
    if examples_rev and all(_roman_to_int(r) == int(n) for r, n in examples_rev):
        query = re.search(r'(?:write|convert)\s+(?:the\s+)?(?:number\s+)?([A-Z]+)', prompt)
        if query:
            rom = query.group(1)
            predicted = str(_roman_to_int(rom))
            cot = f"Roman to decimal conversion.\n\n{rom} = {predicted}\n\n\\boxed{{{predicted}}}"
            return predicted, cot
    return None, None

def solve_text_encryption(prompt, answer):
    is_decrypt = 'decrypt' in prompt.lower()
    examples = re.findall(r'(.+?)\s*->\s*(.+)', prompt)
    query_m = re.search(r'(?:de|en)crypt the following text:\s*(.+?)(?:\n|$)', prompt)
    if not examples or not query_m:
        return None, None
    query = query_m.group(1).strip()
    char_map = {}
    for a, b in examples:
        a, b = a.strip(), b.strip()
        if len(a) != len(b): continue
        for x, y in zip(a, b):
            if x == ' ' and y == ' ': continue
            if x in char_map and char_map[x] != y: return None, None
            char_map[x] = y
    if len(query) == len(answer):
        for c, p in zip(query, answer):
            if c == ' ' and p == ' ': continue
            if c in char_map and char_map[c] != p: return None, None
            char_map[c] = p
    result = ''
    for c in query:
        if c == ' ': result += ' '
        elif c in char_map: result += char_map[c]
        else: return None, None
    direction = "cipher->plain" if is_decrypt else "plain->cipher"
    table = ", ".join(f"'{k}'->'{v}'" for k, v in sorted({c: char_map[c] for c in query if c != ' ' and c in char_map}.items()))
    cot = f"Substitution cipher ({direction}).\n\nMappings: {table}\n\nApplying to '{query}': {result}\n\n\\boxed{{{result}}}"
    return result, cot

# ── Bit Manipulation Solver ──
def _get_bit(s, pos): return int(s[pos])

def _solve_bit_functions(pairs):
    funcs = [None] * 8
    for out_pos in range(8):
        expected = [_get_bit(out, out_pos) for _, out in pairs]
        for in_pos in range(8):
            direct = [_get_bit(inp, in_pos) for inp, _ in pairs]
            if direct == expected: funcs[out_pos] = ('direct', in_pos); break
            if [1-b for b in direct] == expected: funcs[out_pos] = ('not', in_pos); break
        if funcs[out_pos]: continue
        found = False
        for i, j in combinations(range(8), 2):
            bi = [_get_bit(inp, i) for inp, _ in pairs]
            bj = [_get_bit(inp, j) for inp, _ in pairs]
            for name, result in [('xor',[a^b for a,b in zip(bi,bj)]),('and',[a&b for a,b in zip(bi,bj)]),
                                  ('or',[a|b for a,b in zip(bi,bj)]),('nand',[1-(a&b) for a,b in zip(bi,bj)]),
                                  ('nor',[1-(a|b) for a,b in zip(bi,bj)]),('xnor',[1-(a^b) for a,b in zip(bi,bj)])]:
                if result == expected: funcs[out_pos] = (name, i, j); found = True; break
            if found: break
        if funcs[out_pos]: continue
        for i, j, k in combinations(range(8), 3):
            bi = [_get_bit(inp, i) for inp, _ in pairs]
            bj = [_get_bit(inp, j) for inp, _ in pairs]
            bk = [_get_bit(inp, k) for inp, _ in pairs]
            for name, result in [('majority',[1 if (a+b+c)>=2 else 0 for a,b,c in zip(bi,bj,bk)]),
                                  ('choice',[b if a==1 else c for a,b,c in zip(bi,bj,bk)])]:
                if result == expected: funcs[out_pos] = (name, i, j, k); found = True; break
            if found: break
    return funcs

def _apply_bit_func(func, query):
    if func is None: return None
    n = func[0]
    if n == 'direct': return _get_bit(query, func[1])
    if n == 'not': return 1 - _get_bit(query, func[1])
    if n in ('xor','xnor','and','nand','or','nor'):
        a, b = _get_bit(query, func[1]), _get_bit(query, func[2])
        return {'xor':a^b,'xnor':1-(a^b),'and':a&b,'nand':1-(a&b),'or':a|b,'nor':1-(a|b)}[n]
    if n in ('majority','choice'):
        a, b, c = _get_bit(query, func[1]), _get_bit(query, func[2]), _get_bit(query, func[3])
        if n == 'majority': return 1 if (a+b+c)>=2 else 0
        return b if a==1 else c
    return None

def _desc_bit_func(f):
    if f is None: return "unknown"
    n = f[0]
    if n == 'direct': return f"input[{f[1]}]"
    if n == 'not': return f"NOT input[{f[1]}]"
    if n in ('xor','xnor','and','nand','or','nor'): return f"input[{f[1]}] {n.upper()} input[{f[2]}]"
    if n in ('majority','choice'): return f"{n}(input[{f[1]}],input[{f[2]}],input[{f[3]}])"
    return str(f)

def solve_bit_manipulation(prompt, answer):
    pairs = re.findall(r'([01]{8})\s*->\s*([01]{8})', prompt)
    query_m = re.search(r'output for:\s*([01]{8})', prompt)
    if not pairs or not query_m: return None, None
    query = query_m.group(1)
    funcs = _solve_bit_functions(pairs)
    bits = [_apply_bit_func(f, query) for f in funcs]
    if all(b is not None for b in bits):
        predicted = ''.join(str(b) for b in bits)
        if predicted == answer:
            desc = "\n".join(f"  bit[{i}] = {_desc_bit_func(funcs[i])}" for i in range(8))
            cot = f"Analyzing bit transformation from {len(pairs)} examples.\n\n{desc}\n\nApplying to {query}: {predicted}\n\n\\boxed{{{predicted}}}"
            return predicted, cot
    # Scaffold
    desc = "\n".join(f"  bit[{i}] = {_desc_bit_func(funcs[i])}" for i in range(8))
    cot = f"Analyzing bit transformation.\n\n{desc}\n\nResult: {answer}\n\n\\boxed{{{answer}}}"
    return answer, cot

# ── Equation Transformation Solver ──
_EQ_OPS = {
    'add': lambda a,b: str(a+b), 'sub_ab': lambda a,b: str(a-b), 'sub_ba': lambda a,b: str(b-a),
    'abs_diff': lambda a,b: str(abs(a-b)), 'mul': lambda a,b: str(a*b),
    'cat_ab': lambda a,b: str(a)+str(b), 'cat_ba': lambda a,b: str(b)+str(a),
    'div_ab': lambda a,b: str(a//b) if b!=0 else None, 'div_ba': lambda a,b: str(b//a) if a!=0 else None,
    'mod_ab': lambda a,b: str(a%b) if b!=0 else None, 'mod_ba': lambda a,b: str(b%a) if a!=0 else None,
    'xor': lambda a,b: str(a^b), 'and': lambda a,b: str(a&b), 'or': lambda a,b: str(a|b),
    'max': lambda a,b: str(max(a,b)), 'min': lambda a,b: str(min(a,b)),
}

def solve_equation_transformation(prompt, answer):
    lines = prompt.strip().split('\n')
    examples, query = [], None
    for line in lines:
        line = line.strip()
        if 'determine the result for:' in line.lower():
            m = re.search(r'determine the result for:\s*(.+)', line, re.IGNORECASE)
            if m: query = m.group(1).strip()
        elif ' = ' in line and 'wonderland' not in line.lower() and 'transformation' not in line.lower():
            parts = line.split(' = ', 1)
            if len(parts) == 2: examples.append((parts[0].strip(), parts[1].strip()))
    if not examples or not query or len(query) != 5: return None, None
    parsed = []
    for lhs, rhs in examples:
        if len(lhs) != 5: return None, None
        parsed.append((lhs[:2], lhs[2], lhs[3:], rhs))
    q_a, q_op, q_b = query[:2], query[2], query[3:]
    if all(a.isdigit() and b.isdigit() for a,_,b,_ in parsed) and q_a.isdigit() and q_b.isdigit():
        by_op = {}
        for a, op, b, rhs in parsed:
            by_op.setdefault(op, []).append((int(a), int(b), rhs))
        op_map = {}
        for op_char, op_ex in by_op.items():
            for op_name, op_func in _EQ_OPS.items():
                if all(op_func(a, b) == rhs for a, b, rhs in op_ex):
                    op_map[op_char] = op_name; break
        if q_op in op_map:
            predicted = _EQ_OPS[op_map[q_op]](int(q_a), int(q_b))
            if predicted == answer:
                cot = f"Decoding operators from examples.\n\nOperator '{q_op}' = {op_map[q_op]}\n\n{q_a} {op_map[q_op]} {q_b} = {predicted}\n\n\\boxed{{{predicted}}}"
                return predicted, cot
    cot = f"Analyzing transformation rules from examples.\n\nApplying to {query}: {answer}\n\n\\boxed{{{answer}}}"
    return answer, cot

# ═══════════════════════════════════════════════════════════════════════════════
# PUZZLE TYPE DETECTION + DATA BUILDING
# ═══════════════════════════════════════════════════════════════════════════════

def detect_puzzle_type(prompt):
    p = prompt.lower()
    if 'gravitational' in p or 'd = 0.5*g*t^2' in p: return 'gravity'
    if 'unit' in p and 'conver' in p and 'becomes' in p: return 'unit'
    if re.search(r'\d+\s*->\s*[A-Z]+|[A-Z]+\s*->\s*\d+', prompt) and ('convert' in p or 'roman' in p): return 'base'
    if ('encrypt' in p or 'decrypt' in p) and '->' in prompt: return 'text'
    if re.search(r'[01]{8}\s*->', prompt) and 'output for:' in p: return 'bit'
    if 'transformation' in p and '=' in prompt and 'determine the result for:' in p: return 'equation'
    return 'unknown'

SOLVER_MAP = {
    'gravity': solve_gravity, 'unit': solve_unit_conversion, 'base': solve_base_conversion,
    'text': solve_text_encryption, 'bit': solve_bit_manipulation, 'equation': solve_equation_transformation,
}

stats = {'solved': 0, 'fallback': 0, 'total': 0}
type_counts = {}

def build_training_text(example):
    prompt, answer = example["prompt"], str(example["answer"])
    ptype = detect_puzzle_type(prompt)
    type_counts[ptype] = type_counts.get(ptype, 0) + 1
    stats['total'] += 1

    cot = None
    if ptype in SOLVER_MAP:
        _, cot = SOLVER_MAP[ptype](prompt, answer)
    if cot:
        stats['solved'] += 1
        assistant_msg = cot
    else:
        stats['fallback'] += 1
        assistant_msg = f"After analyzing the pattern from the examples:\n\n\\boxed{{{answer}}}"

    user_msg = prompt + BOXED_INSTRUCTION
    try:
        messages = [{"role": "user", "content": user_msg}, {"role": "assistant", "content": assistant_msg}]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    except Exception:
        text = f"<|im_start|>user\n{user_msg}<|im_end|>\n<|im_start|>assistant\n{assistant_msg}<|im_end|>"
    return {"text": text}

# Build SFT dataset
hf_dataset = hf_dataset.map(build_training_text, remove_columns=hf_dataset.column_names)
print(f"SFT dataset: {len(hf_dataset)} examples")
print(f"  Solver CoT: {stats['solved']}, Fallback: {stats['fallback']}")
print(f"  Types: {dict(sorted(type_counts.items(), key=lambda x: -x[1]))}")

# Build GRPO dataset (prompt-only, no gold answer in text)
grpo_sub = train_pd.sample(n=min(GRPO_SUBSAMPLE_SIZE, len(train_pd)), random_state=42)
grpo_records = []
for _, row in grpo_sub.iterrows():
    grpo_records.append({
        "prompt": [{"role": "user", "content": row["prompt"] + BOXED_INSTRUCTION}],
        "ground_truth": str(row["answer"]),
    })
grpo_dataset = Dataset.from_list(grpo_records)
print(f"GRPO dataset: {len(grpo_dataset)} prompts")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# MODEL LOADING + SFT TRAINING
# ═══════════════════════════════════════════════════════════════════════════════

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, device_map="auto", trust_remote_code=True, dtype=torch.bfloat16
)
print(f"Model loaded. Vocab size: {len(tokenizer)}")

# Disable fast path (Triton issues with Nemotron)
for name, mod in sys.modules.items():
    if "modeling_nemotron_h" in name:
        mod.is_fast_path_available = False
        print(f"Patched {name}: is_fast_path_available = False")

if SKIP_SFT:
    # Load adapter from checkpoint
    print(f"SKIP_SFT=True — loading adapter from {SFT_CHECKPOINT_PATH}")
    model = PeftModel.from_pretrained(model, SFT_CHECKPOINT_PATH, is_trainable=True)
    model.print_trainable_parameters()
    print("SFT adapter loaded. Ready for GRPO.")
else:
    # Apply fresh LoRA
    lora_config = LoraConfig(
        r=LORA_RANK,
        lora_alpha=16,
        target_modules="all-linear",
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    # Triton compiler fix
    import triton.backends.nvidia.compiler as nv_compiler
    os.environ["TRITON_PTXAS_BLACKWELL_PATH"] = "/tmp/ptxas-blackwell"
    nv_compiler.get_ptxas_version = lambda arch: "12.0"

    # SFT Training
    training_args = SFTConfig(
        output_dir=OUTPUT_DIR,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=NUM_EPOCHS,
        learning_rate=LR,
        logging_steps=5,
        bf16=True,
        max_grad_norm=1.0,
        optim="adamw_torch",
        lr_scheduler_type="cosine",
        warmup_ratio=0.1,
        save_strategy="no",
        report_to="none",
        dataset_text_field="text",
        max_length=MAX_SEQ_LEN,
        packing=False,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": True},
    )

    trainer = SFTTrainer(
        model=model,
        train_dataset=hf_dataset,
        processing_class=tokenizer,
        args=training_args,
    )

    print("Starting SFT training...")
    trainer.train()
    print("SFT training complete.")

    # Save SFT adapter (download & upload as dataset to skip SFT later)
    trainer.model.save_pretrained(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)
    print(f"SFT adapter saved to {OUTPUT_DIR}/")

    # Free SFT trainer memory
    del trainer
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# GRPO REWARD FUNCTIONS
# ═══════════════════════════════════════════════════════════════════════════════

def _normalize_answer(s):
    s = s.strip()
    try:
        f = float(s)
        return str(int(f)) if f == int(f) else str(f)
    except (ValueError, OverflowError):
        return s

def _extract_boxed(content):
    m = re.search(r'\\boxed\{([^}]*)\}', content, re.DOTALL)
    if m: return m.group(1).strip()
    m = re.search(r'boxed\{([^}]*)\}', content, re.DOTALL)
    if m: return m.group(1).strip()
    return None

def _get_content(completion):
    if isinstance(completion, list):
        return completion[-1]["content"] if completion else ""
    return completion

def cosine_reward(completions, ground_truth, **kwargs):
    """Correct + short -> ~1.0, Correct + long -> ~0.1, Wrong -> negative."""
    rewards = []
    for completion, gt in zip(completions, ground_truth):
        content = _get_content(completion)
        extracted = _extract_boxed(content)
        progress = min(len(content) / max(GRPO_MAX_COMPLETION, 1), 1.0)
        cos_scale = 0.5 * (1.0 + math.cos(math.pi * progress))
        if extracted is not None and _normalize_answer(extracted) == _normalize_answer(gt):
            rewards.append(0.1 + 0.9 * cos_scale)
        elif extracted is not None:
            rewards.append(-0.1 - 0.9 * (1.0 - cos_scale))
        else:
            rewards.append(-0.5 * progress)
    return rewards

def format_reward(completions, **kwargs):
    """1.0 if \\boxed{} present, 0.0 otherwise."""
    return [1.0 if _extract_boxed(_get_content(c)) is not None else 0.0 for c in completions]

def length_reward(completions, **kwargs):
    """Linear penalty 0.0 -> -1.0 as completion approaches max length."""
    return [-min(len(_get_content(c)) / max(GRPO_MAX_COMPLETION, 1), 1.0) for c in completions]

# Sanity check
_t = cosine_reward(["\\boxed{42}", "no box", "\\boxed{99}" + " "*800], ["42","42","42"])
print(f"Reward sanity: {[f'{r:.3f}' for r in _t]}  (expect ~[1.0, 0.0, ~-0.7])")



In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# GRPO TRAINING
# ═══════════════════════════════════════════════════════════════════════════════

gc.collect()
torch.cuda.empty_cache()

model.train()
tokenizer.padding_side = "left"

grpo_config = GRPOConfig(
    output_dir="./grpo-checkpoints",
    num_generations=GRPO_NUM_GENERATIONS,
    generation_batch_size=GRPO_NUM_GENERATIONS,
    max_completion_length=GRPO_MAX_COMPLETION,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    num_train_epochs=GRPO_EPOCHS,
    learning_rate=GRPO_LR,
    temperature=GRPO_TEMPERATURE,
    beta=0.0,
    loss_type="grpo",
    logging_steps=2,
    max_grad_norm=0.1,
    weight_decay=0.1,
    optim="adamw_torch",
    lr_scheduler_type="cosine",
    warmup_steps=5,
    save_strategy="no",
    report_to="none",
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": True},
    remove_unused_columns=False,
)

model.warnings_issued = {}

grpo_trainer = GRPOTrainer(
    model=model,
    reward_funcs=[cosine_reward, format_reward, length_reward],
    train_dataset=grpo_dataset,
    processing_class=tokenizer,
    args=grpo_config,
)

print(f"GRPO: {GRPO_NUM_GENERATIONS} gen/prompt, temp={GRPO_TEMPERATURE}, lr={GRPO_LR}")
print(f"GRPO dataset: {len(grpo_dataset)} prompts")
print("Starting GRPO training...")
grpo_trainer.train()
print("GRPO training complete.")

# Save final adapter
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Final adapter (SFT+GRPO) saved to {OUTPUT_DIR}/")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SUBMISSION
# ═══════════════════════════════════════════════════════════════════════════════

# Cleanup
for d in ["./grpo-checkpoints", os.path.join(OUTPUT_DIR, "checkpoint-*")]:
    if os.path.exists(d):
        shutil.rmtree(d, ignore_errors=True)

zip_path = "/kaggle/working/submission.zip"
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in os.listdir(OUTPUT_DIR):
        fpath = os.path.join(OUTPUT_DIR, fname)
        zf.write(fpath, fname)

print(f"Created {zip_path} ({os.path.getsize(zip_path)/1024/1024:.1f} MB)")
with zipfile.ZipFile(zip_path, 'r') as zf:
    print(f"Contents: {zf.namelist()}")
    assert "adapter_config.json" in zf.namelist()
print("submission.zip ready!")